In [ ]:
import torch
from models.handler import Handler
from models.model import ResnetClassifier
from models.loss import FairLoss
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torchvision import transforms
from torchvision.datasets import FashionMNIST, CIFAR10, CIFAR100, MNIST
from models.noise import InstanceDependentNoiseAdder

# Prepare

In [ ]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
device = torch.device(device)
device

In [ ]:
train_dataset = CIFAR10(root='data', train=True, download=False)
# noise_adder = InstanceDependentNoiseAdder(train_dataset, 0.2, 10)
# noise_adder.add_noise()
test_dataset = CIFAR10(root='data', train=False, download=False)
transform = transforms.Compose([
                                transforms.ToTensor(),
                                transforms.RandomCrop(32, padding=4),
                                transforms.RandomGrayscale(p=0.5),
                                transforms.RandomHorizontalFlip(p=0.5),
                                transforms.RandomVerticalFlip(p=0.5),
                                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

In [ ]:
model = ResnetClassifier()
model = model.to(device)
torch.save(model.state_dict(), 'model.pth')

# CE

In [ ]:
model = ResnetClassifier()
model = model.to(device)
model.load_state_dict(torch.load('model.pth'))
critertion = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)
handler = Handler(model, device, critertion, optimizer, train_dataset, test_dataset, transform)

In [ ]:
handler.train(400)

In [ ]:
handler.test()

# Fair

In [ ]:
model = ResnetClassifier()
model = model.to(device)
model.load_state_dict(torch.load('model.pth'))
critertion = FairLoss()
optimizer = Adam(model.parameters(), lr=0.001)
handler = Handler(model, device, critertion, optimizer, train_dataset, test_dataset, transform)

In [ ]:
handler.train(400)

In [ ]:
handler.test()

# Compare

In [ ]:
from models.repeater import Repeater

repeater = Repeater()
repeater.repeat(5, 200)

 22%|██▎       | 45/200 [10:35<36:49, 14.26s/it]